In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input
import numpy as np
import matplotlib.pyplot as plt
import random

# ==========================================
# 1. DATA PREPARATION (Rich Text Generation)
# ==========================================
print("Loading MNIST and generating sentences...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize Images
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
# Add channel dimension (28, 28, 1) for Conv2D
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# --- Generate Rich Captions ---
# We create diverse templates so the model has to learn actual language, not just labels.
templates = [
    "this is a handwritten {}",
    "the number shown is {}",
    "a digit representing {}",
    "picture of a {}"
]
# Map digits to words
digit_words = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']

def create_captions(labels):
    captions = []
    for label in labels:
        digit_str = digit_words[label]
        # Pick a random sentence structure for variety
        template = random.choice(templates)
        captions.append(template.format(digit_str))
    return np.array(captions)

# Generate sentences
t_train_text = create_captions(y_train)
t_test_text = create_captions(y_test)

print(f"Example Caption 1: '{t_train_text[0]}'")
print(f"Example Caption 2: '{t_train_text[1]}'")

# --- Text Tokenization ---
MAX_VOCAB = 100  # We have a small vocabulary
SEQ_LEN = 8      # Max length of sentence

vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_sequence_length=SEQ_LEN,
    standardize='lower_and_strip_punctuation'
)
vectorizer.adapt(t_train_text)

# Convert sentences to integers
t_train_seq = vectorizer(t_train_text).numpy()
t_test_seq = vectorizer(t_test_text).numpy()

vocab_size = vectorizer.vocabulary_size()
print(f"Data Ready. Vocab Size: {vocab_size}")

# ==========================================
# 2. DEFINE THE MULTIMODAL MODEL
# ==========================================
LATENT_DIM = 64 # Size of the shared "Brain" (z)

# --- A. Image Encoder (Vision) ---
img_input = Input(shape=(28, 28, 1), name="Image_Input")
x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(img_input)
x = layers.MaxPooling2D((2, 2), padding='same')(x) # 14x14
x = layers.Conv2D(8, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2), padding='same')(x) # 7x7
x = layers.Flatten()(x)
img_encoded = layers.Dense(64, activation='relu')(x)

# --- B. Text Encoder (Language) ---
txt_input = Input(shape=(SEQ_LEN,), name="Text_Input")
y = layers.Embedding(vocab_size, 32)(txt_input)
y = layers.LSTM(64)(y) # LSTM reads the sequence
txt_encoded = layers.Dense(64, activation='relu')(y)

# --- C. Fusion (The Joint Brain) ---
concat = layers.concatenate([img_encoded, txt_encoded])
z = layers.Dense(LATENT_DIM, activation='relu', name="Joint_Z")(concat)

# --- D. Image Decoder (Reconstruction) ---
d_x = layers.Dense(7*7*8, activation='relu')(z)
d_x = layers.Reshape((7, 7, 8))(d_x)
d_x = layers.Conv2DTranspose(8, (3, 3), strides=2, activation='relu', padding='same')(d_x)
d_x = layers.Conv2DTranspose(16, (3, 3), strides=2, activation='relu', padding='same')(d_x)
img_output = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same', name="Image_Output")(d_x)

# --- E. Text Decoder (Generation) ---
# To generate text, we repeat the concept 'z' for every step of the sequence
d_y = layers.RepeatVector(SEQ_LEN)(z)
d_y = layers.LSTM(64, return_sequences=True)(d_y)
# Output probability for every word in vocab at every step
txt_output = layers.Dense(vocab_size, activation='softmax', name="Text_Output")(d_y)

# --- Build & Compile ---
autoencoder = models.Model(inputs=[img_input, txt_input], outputs=[img_output, txt_output])

autoencoder.compile(
    optimizer='adam',
    loss={'Image_Output': 'mse', 'Text_Output': 'sparse_categorical_crossentropy'},
    loss_weights=[1.0, 1.0]
)

# ==========================================
# 3. TRAIN
# ==========================================
print("\nTraining Multimodal Autoencoder...")
# We train for just 5 epochs to keep it fast for testing
autoencoder.fit(
    [x_train, t_train_seq],
    [x_train, t_train_seq],
    epochs=5,
    batch_size=128,
    validation_data=([x_test, t_test_seq], [x_test, t_test_seq])
)

# ==========================================
# 4. VISUALIZATION
# ==========================================
print("\n--- Visualizing Results ---")

# Helper to convert numbers back to words
vocab = vectorizer.get_vocabulary()
def decode_sequence(seq):
    return " ".join([vocab[i] for i in seq if i != 0])

# Predict on Test Data
pred_imgs, pred_texts = autoencoder.predict([x_test[:10], t_test_seq[:10]])

plt.figure(figsize=(15, 8))
for i in range(5):
    # Original Image
    ax = plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    ax.set_title("Original")
    plt.axis("off")

    # Reconstructed Image
    ax = plt.subplot(3, 5, i + 1 + 5)
    plt.imshow(pred_imgs[i].reshape(28, 28), cmap='gray')
    ax.set_title("Reconstructed")
    plt.axis("off")

    # Text Comparison
    # We choose the word with highest probability at each step (argmax)
    generated_seq = np.argmax(pred_texts[i], axis=-1)
    original_text = decode_sequence(t_test_seq[i])
    generated_text = decode_sequence(generated_seq)

    # Display Text below
    ax = plt.subplot(3, 5, i + 1 + 10)
    plt.text(0, 0.8, f"Orig: {original_text}", fontsize=10)
    plt.text(0, 0.2, f"Pred: {generated_text}", fontsize=10, color="blue")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 5. ROBUSTNESS TEST: MISSING MODALITIES
# ==========================================
print("\n--- ROBUSTNESS TEST: Handling Missing Data ---")

# Let's pick a specific example, e.g., a '7'
# Find the first '7' in the test set
target_idx = np.where(y_test == 7)[0][0]

# Prepare the inputs
real_img = x_test[target_idx:target_idx+1]      # Shape (1, 28, 28, 1)
real_txt = t_test_seq[target_idx:target_idx+1]  # Shape (1, 8)
empty_img = np.zeros_like(real_img)             # All Black Image
empty_txt = np.zeros_like(real_txt)             # Empty Text Sequence

# --- SCENARIO A: BLIND GENERATION (Text -> Image) ---
# We feed the Text, but the Image input is 100% empty (black).
# The model must use the "concept" of the text to draw the image.
print(f"Scenario A: Input Text='{decode_sequence(real_txt[0])}' | Input Image=BLANK")
gen_img_from_text, _ = autoencoder.predict([empty_img, real_txt])

# --- SCENARIO B: MUTE GENERATION (Image -> Text) ---
# We feed the Image, but the Text input is empty.
# The model must use the "concept" of the image to write the text.
print(f"Scenario B: Input Image=See Below | Input Text=EMPTY")
_, gen_txt_from_img = autoencoder.predict([real_img, empty_txt])

# --- VISUALIZE THE ROBUSTNESS ---
plt.figure(figsize=(12, 6))

# 1. The Original (Ground Truth)
ax = plt.subplot(1, 3, 1)
plt.imshow(real_img.reshape(28, 28), cmap='gray')
ax.set_title("Original Input")
plt.axis("off")

# 2. Scenario A Result
ax = plt.subplot(1, 3, 2)
plt.imshow(gen_img_from_text.reshape(28, 28), cmap='gray')
ax.set_title("Generated from ONLY Text\n(Image Input was 0)")
plt.axis("off")

# 3. Scenario B Result
ax = plt.subplot(1, 3, 3)
# Show the image used as input
plt.imshow(real_img.reshape(28, 28), cmap='gray')
# Decode the generated text
gen_seq = np.argmax(gen_txt_from_img[0], axis=-1)
gen_sentence = decode_sequence(gen_seq)
ax.set_title(f"Generated from ONLY Image\nPred: '{gen_sentence}'")
plt.axis("off")

plt.tight_layout()
plt.show()
